# Phase 159 CryptoBench Dataset Inspection

This notebook automatically locates the Phase 159 .pt file under /kaggle/input.
It does not train or modify the dataset.


In [ ]:
!pip install -q torch-geometric


In [ ]:
from pathlib import Path
import torch

files = list(Path('/kaggle/input').rglob('phase159_cryptobench_anmmsf_graphs.pt'))

print('MATCHES:')
for f in files:
    print(f)

if not files:
    print('\nALL .PT FILES:')
    for f in Path('/kaggle/input').rglob('*.pt'):
        print(f)
    raise FileNotFoundError('Phase 159 .pt file was not found under /kaggle/input')

p = files[0]
print('\nUSING:', p)
print('SIZE GB:', p.stat().st_size / 1024**3)


In [ ]:
x = torch.load(p, map_location='cpu', weights_only=False)

print('TOP-LEVEL TYPE:', type(x))

if isinstance(x, dict):
    print('\nTOP-LEVEL KEYS:')
    for k, v in x.items():
        if hasattr(v, 'shape'):
            print(f'  {k}: shape={tuple(v.shape)}, dtype={getattr(v, "dtype", None)}')
        elif isinstance(v, (list, tuple)):
            print(f'  {k}: {type(v).__name__}, len={len(v)}')
        else:
            print(f'  {k}: {type(v).__name__}: {str(v)[:500]}')


In [ ]:
def describe(name, obj, indent=0):
    prefix = ' ' * indent
    print(f'{prefix}{name}: {type(obj)}')

    if isinstance(obj, dict):
        for k, v in obj.items():
            if hasattr(v, 'shape'):
                print(f'{prefix}  {k}: shape={tuple(v.shape)}, dtype={getattr(v, "dtype", None)}')
            elif isinstance(v, (list, tuple)):
                print(f'{prefix}  {k}: {type(v).__name__}, len={len(v)}')
            else:
                print(f'{prefix}  {k}: {type(v).__name__}: {str(v)[:300]}')
    elif isinstance(obj, (list, tuple)):
        print(f'{prefix}  length={len(obj)}')
        if len(obj):
            describe('[0]', obj[0], indent + 2)

describe('DATASET', x)


In [ ]:
if isinstance(x, dict) and 'dataset' in x:
    ds = x['dataset']
elif isinstance(x, (list, tuple)):
    ds = x
else:
    ds = None

if ds is None:
    print('No dataset list found at the expected top level.')
else:
    print('NUMBER OF GRAPHS:', len(ds))

    if len(ds):
        d = ds[0]
        print('\nFIRST GRAPH TYPE:', type(d))
        print('\nFIRST GRAPH:')
        print(d)
        print('\nGRAPH ATTRIBUTES:')
        print(d.keys())

        print('\nATTRIBUTE DETAILS:')
        for key in d.keys():
            value = d[key]
            print(
                f'  {key}: '
                f'type={type(value)}, '
                f'shape={getattr(value, "shape", None)}, '
                f'dtype={getattr(value, "dtype", None)}'
            )


In [ ]:
from collections import Counter
        
if ds is not None and len(ds):
    print('SPLIT COUNTS:')
    print(Counter(str(getattr(d, 'split', 'MISSING')) for d in ds))
        
    counts = [d.x.shape[0] for d in ds if hasattr(d, 'x')]
    if counts:
        import numpy as np
        print('\nNODE COUNTS:')
        print('min:', min(counts))
        print('max:', max(counts))
        print('mean:', float(np.mean(counts)))
        print('median:', float(np.median(counts)))
        print('\nNODE FEATURE SHAPES:')
        print(sorted(set(tuple(d.x.shape) for d in ds if hasattr(d, 'x'))))
        
    print('\nLABEL COUNTS:')
    if all(hasattr(d, 'y') for d in ds):
        pos = sum(int(d.y.sum()) for d in ds)
        total = sum(int(d.y.numel()) for d in ds)
        print('positive:', pos)
        print('negative:', total - pos)
        print('positive fraction:', pos / total if total else None)


In [ ]:
print('\nTOP-LEVEL METADATA:')
if isinstance(x, dict):
    for k, v in x.items():
        if k != 'dataset':
            print('\n---', k, '---')
            print(str(v)[:5000])
